In [1]:
import sys
import numpy as np
import xf_midi
from settings import RWC_DATASET_PATH, LA_DATASET_PATH, NOTTINGHAM_DATASET_PATH
import os
from joblib import Parallel, delayed
import torch
import shutil
import json
import pretty_midi
import argparse

sys.path.append("/home/bowen.zheng/Documents")
from StreamMUSE.m2a_transformer import RoFormerSymbolicTransformer, EOS_TOKEN, PAD_TOKEN
# from preprocess_midi2pt_dataset import preprocess_midi
print(os.getcwd())

/home/bowen.zheng/Documents/StreamMUSE/.venv/lib/python3.10/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bowen.zheng/Documents/StreamMUSE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-11 11:39:50.210571: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-11 11:39:50.591666: E external/local_xla/xla/strea

/home/bowen.zheng/Documents/StreamMUSE/preprocess


In [4]:
notebook_dir = os.path.dirname(os.path.abspath('test.ipynb'))
midi_path = os.path.join(notebook_dir, '../input/mel/001.mid')
print(midi_path)
print(os.path.exists(midi_path))

/home/bowen.zheng/Documents/StreamMUSE/preprocess/../input/mel/001.mid
True


In [5]:
tokenize_dict = {'<sos>': 0, '<eos>': 1, '<pad>': 2}
tokenize_count = [-1, -1, -1]

DURATION_TEMPLATES = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048, 3072, 4096])

def preprocess_midi(midi_path, max_polyphony, beat_div=4, ins_ids='all'):
    print(midi_path)
    try:
        midi = xf_midi.XFMidi(midi_path, constant_tempo=60.0 / beat_div)
    except Exception as e:
        print(f"Error processing {midi_path}: Invalid MIDI file. Error: {e}")
        return None
    print(midi)
    midi_end_time = int(midi.get_end_time())
    print("midi_end_time:",midi_end_time)
    if midi_end_time <= 0:
        return None
    if not isinstance(ins_ids, list):
        ins_ids = [ins_ids]
    duration_boundaries = (DURATION_TEMPLATES[1:] + DURATION_TEMPLATES[:-1]) / 2
    print("durantion boundaries:", duration_boundaries)
    min_pitch = 127
    max_pitch = 0
    result_rolls = []
    print("ins_ids:", ins_ids)
    for ins_id in ins_ids:
        has_any_note = False
        rolls = np.full((midi_end_time, max_polyphony, 3), dtype=np.uint8, fill_value=255)
        polyphony_counts = np.zeros(midi_end_time, dtype=np.uint8)
        for i, ins in enumerate(midi.instruments):
            print(i, ins)
            program = ins.program
            if ins.is_drum:
                program = 127
            for note in ins.notes:
                start_time = int(round(note.start))
                end_time = int(round(note.end))
                if start_time >= 0 and end_time < midi_end_time and polyphony_counts[start_time] < max_polyphony:
                    if ins.is_drum:
                        duration = 0
                    else:
                        duration = np.searchsorted(duration_boundaries, end_time - start_time) #为了做四舍五入的quantize
                        min_pitch = min(min_pitch, note.pitch)
                        max_pitch = max(max_pitch, note.pitch)
                    add_note = False
                    if ins_id == 'all':
                        add_note = True
                    elif isinstance(ins_id, int):
                        raise NotImplementedError
                    elif isinstance(ins_id, str):
                        if '-' in ins_id:
                            task, num = ins_id.split('-')
                            num = int(num)
                            if task == 'track':
                                add_note = i == num
                            elif task == 'upto':
                                add_note = i <= num
                            elif task == 'from':
                                add_note = i >= num
                            elif task == 'notrack':
                                add_note = i != num
                            else:
                                raise NotImplementedError
                        elif ins_id == 'drum':
                            add_note = ins.is_drum
                        elif ins_id == 'nondrum':
                            add_note = not ins.is_drum
                        elif ins_id == 'empty':
                            add_note = False
                        else:
                            raise NotImplementedError
                    else:
                        raise NotImplementedError
                    if add_note:
                        has_any_note = True
                        rolls[start_time, polyphony_counts[start_time]] = [program, note.pitch, duration]
                        # [program, pitch, duration]
                        polyphony_counts[start_time] += 1
        if not has_any_note and ins_id != 'empty':
            return None  # invalid midi file
        for i in range(midi_end_time):
            # Sort notes by ins first, then by pitch, then by duration
            rolls[i, :polyphony_counts[i]] = rolls[i, :polyphony_counts[i]][np.lexsort((rolls[i, :polyphony_counts[i], 2], rolls[i, :polyphony_counts[i], 1], rolls[i, :polyphony_counts[i], 0]))]
            if polyphony_counts[i] < max_polyphony:
                rolls[i, polyphony_counts[i], 0] = 254  # EOS token
        result_rolls.append(rolls)
        print("polyphony_counts:", polyphony_counts[10:100])
    result_rolls = np.concatenate(result_rolls, axis=1) 
    print(result_rolls)
    # Get song-level pitch shift range
    pitch_shift_max = 127 - max_pitch
    pitch_shift_min = -min_pitch
    print(midi_path, ": final", torch.tensor(result_rolls.reshape(midi_end_time, -1)).shape, torch.tensor([pitch_shift_min, pitch_shift_max], dtype=torch.int8).shape)
    return torch.tensor(result_rolls.reshape(midi_end_time, -1)), torch.tensor([pitch_shift_min, pitch_shift_max], dtype=torch.int8)


In [6]:
test_res1_mel = preprocess_midi(midi_path, 4)
test_res1_acc = preprocess_midi(midi_path.replace("mel", "acc"), 4)

/home/bowen.zheng/Documents/StreamMUSE/preprocess/../input/mel/001.mid
midi_end_time: 1093
durantion boundaries: [1.500e+00 2.500e+00 3.500e+00 5.000e+00 7.000e+00 1.000e+01 1.400e+01
 2.000e+01 2.800e+01 4.000e+01 5.600e+01 8.000e+01 1.120e+02 1.600e+02
 2.240e+02 3.200e+02 4.480e+02 6.400e+02 8.960e+02 1.280e+03 1.792e+03
 2.560e+03 3.584e+03]
ins_ids: ['all']
0 Instrument(program=0, is_drum=False, name="MELODY")
polyphony_counts: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 1 0
 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0]
[[[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 ...

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 

In [7]:
print("test_res1_mel:", test_res1_mel[0], test_res1_mel[1])
print("test_res1_mel shape:", test_res1_mel[0].shape, test_res1_mel[1].shape)

print("test_res1_acc:", test_res1_acc[0], test_res1_acc[1])
print("test_res1_acc shape:", test_res1_acc[0].shape, test_res1_acc[1].shape)

test_res1_mel: tensor([[254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8) tensor([-61,  57], dtype=torch.int8)
test_res1_mel shape: torch.Size([1093, 12]) torch.Size([2])
test_res1_acc: tensor([[254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8) tensor([-39,  40], dtype=torch.int8)
test_res1_acc shape: torch.Size([1163, 12]) torch.Size([2])


In [8]:
def decompress(model, byte_arr_mel, byte_arr_acc):
    x = torch.tensor(byte_arr_mel).unsqueeze(0)
    # x = x.cuda()
    y = torch.tensor(byte_arr_acc).unsqueeze(0)
    # y = y.cuda()
    return model.preprocess(x, pitch_shift=torch.zeros(1, dtype=torch.int8).cuda(), y=y)

In [9]:
model_path = "/home/bowen.zheng/Documents/StreamMUSE/results/ModelBaseline/cp_transformer_909+ac+1k7_trackemb_interleavepos_v0.2_large_batch_40_schedule.epoch=00.val_loss=0.90296.ckpt"
model = RoFormerSymbolicTransformer.load_from_checkpoint(model_path, large=True)

/home/bowen.zheng/Documents/StreamMUSE/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [10]:
test_res2_mel, test_res2_acc = decompress(model, test_res1_mel[0], test_res1_acc[0])

/tmp/ipykernel_1027824/1366587090.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(byte_arr_mel).unsqueeze(0)


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
print(torch.cuda.is_available())
print(torch.version.cuda)  # 应该输出类似 11.8、12.1 等
print(torch.backends.cudnn.enabled)  # 应该为 True

False
12.6
True


In [6]:
y = [
    torch.randint(3200, 3300, (1, 8)),  # [1,8]
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 8)),
    torch.randint(3200, 3300, (1, 1)),  # [1,1]
    torch.randint(3200, 3300, (1, 3)),  # [1,3]
]
max_subseq_len = 8
y_fixed = []
for t in y:
    if t.shape[1] < max_subseq_len:
        pad_len = max_subseq_len - t.shape[1]
        pad = torch.full((1, pad_len), PAD_TOKEN, dtype=t.dtype, device=t.device)
        t = torch.cat([t, pad], dim=1)
    y_fixed.append(t)
print("y_fixed:")
for i, t in enumerate(y_fixed):
    print(f"y_fixed[{i}].shape = {t.shape}, values = {t}")

y_fixed:
y_fixed[0].shape = torch.Size([1, 8]), values = tensor([[3262, 3230, 3255, 3212, 3233, 3213, 3251, 3240]])
y_fixed[1].shape = torch.Size([1, 8]), values = tensor([[3273, 3280, 3208, 3267, 3223, 3232, 3276, 3234]])
y_fixed[2].shape = torch.Size([1, 8]), values = tensor([[3286, 3276, 3290, 3218, 3264, 3286, 3263, 3200]])
y_fixed[3].shape = torch.Size([1, 8]), values = tensor([[3220, 3292, 3259, 3223, 3200, 3298, 3248, 3221]])
y_fixed[4].shape = torch.Size([1, 8]), values = tensor([[3296, 3204, 3269, 3219, 3262, 3282, 3271, 3245]])
y_fixed[5].shape = torch.Size([1, 8]), values = tensor([[3266, 3279, 3288, 3212, 3246, 3299, 3273, 3268]])
y_fixed[6].shape = torch.Size([1, 8]), values = tensor([[3218, 3254, 3229, 3213, 3211, 3202, 3213, 3214]])
y_fixed[7].shape = torch.Size([1, 8]), values = tensor([[3289, 3283, 3255, 3252, 3254, 3283, 3270, 3266]])
y_fixed[8].shape = torch.Size([1, 8]), values = tensor([[3253, 3259, 3210, 3262, 3296, 3264, 3258, 3210]])
y_fixed[9].shape = torch.Siz